<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 理解 Embedding 层与 Linear 层之间的区别

- PyTorch 中的 embedding 层与执行矩阵乘法的 linear 层实现的功能相同；我们使用 embedding 层的原因是计算效率更高
- 我们将使用 PyTorch 中的代码示例逐步了解这种关系

In [1]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.3.1


<br>
&nbsp;

## 使用 nn.Embedding

In [2]:
# Suppose we have the following 3 training examples,
# which may represent token IDs in a LLM context
idx = torch.tensor([2, 3, 1])

# The number of rows in the embedding matrix can be determined
# by obtaining the largest token ID + 1.
# If the highest token ID is 3, then we want 4 rows, for the possible
# token IDs 0, 1, 2, 3
num_idx = max(idx)+1

# The desired embedding dimension is a hyperparameter
out_dim = 5

- 让我们实现一个简单的 embedding 层：

In [3]:
# We use the random seed for reproducibility since
# weights in the embedding layer are initialized with
# small random values
torch.manual_seed(123)

embedding = torch.nn.Embedding(num_idx, out_dim)

我们可以选择查看一下 embedding 权重：

In [4]:
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)

- 然后我们可以使用 embedding 层来获取 ID 为 1 的训练样本的向量表示：

In [5]:
embedding(torch.tensor([1]))

tensor([[ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- 下面是底层工作原理的可视化：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/1.png" width="400px">

**图 2.1**：Embedding 层本质上是一个查找表操作。给定一个 token ID（如 1），它从权重矩阵中直接查找对应行，返回该行的向量表示。

```
Token ID 1  ──查找──▶  Embedding 权重矩阵  ──▶  返回第 1 行向量
                         [第0行] [a0, a1, a2, a3]
                         [第1行] [b0, b1, b2, b3] ◀── 查找结果
                         [第2行] [c0, c1, c2, c3]
                         [第3行] [d0, d1, d2, d3]
```

> **核心结论**：Embedding 层不是在做复杂的数学运算，而是直接从权重表中"查表"，效率极高。

- 类似地，我们可以使用 embedding 层来获取 ID 为 2 的训练样本的向量表示：

In [6]:
embedding(torch.tensor([2]))

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315]],
       grad_fn=<EmbeddingBackward0>)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/2.png" width="400px">

**图 2.2**：与上一张图相同的概念，只是查找的是 Token ID 2 对应的行向量。

```
Token ID 2  ──查找──▶  Embedding 权重矩阵  ──▶  返回第 2 行向量
                         [第0行] [a0, a1, a2, a3]
                         [第1行] [b0, b1, b2, b3]
                         [第2行] [c0, c1, c2, c3] ◀── 查找结果
                         [第3行] [d0, d1, d2, d3]
```

> **核心结论**：无论输入哪个 token ID，Embedding 层都能快速返回对应的行向量。

- 现在，让我们将之前定义的所有训练样本进行转换：

In [7]:
idx = torch.tensor([2, 3, 1])
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- 在底层，它仍然是相同的查找概念：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/3.png" width="450px">

**图 2.3**：当输入多个 token ID 时，Embedding 层对每个 ID 进行独立的查找，返回对应的行向量组合。

```
Token IDs [2, 3, 1]  ──批量查找──▶  返回:
  ID 2 → [c0, c1, c2, c3]
  ID 3 → [d0, d1, d2, d3]
  ID 1 → [b0, b1, b2, b3]
```

> **核心结论**：Embedding 层支持批量查找，一次处理多个 token ID，高效地返回多行向量。

<br>
&nbsp;

## 使用 nn.Linear

- 现在，我们将演示上面的 embedding 层在 PyTorch 中对 one-hot 编码表示执行的操作与 `nn.Linear` 层完全相同
- 首先，让我们将 token ID 转换为 one-hot 表示：

In [8]:
onehot = torch.nn.functional.one_hot(idx)
onehot

tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])

- 接下来，我们初始化一个 `Linear` 层，它执行矩阵乘法 $X W^\top$：

In [9]:
torch.manual_seed(123)
linear = torch.nn.Linear(num_idx, out_dim, bias=False)
linear.weight

Parameter containing:
tensor([[-0.2039,  0.0166, -0.2483,  0.1886],
        [-0.4260,  0.3665, -0.3634, -0.3975],
        [-0.3159,  0.2264, -0.1847,  0.1871],
        [-0.4244, -0.3034, -0.1836, -0.0983],
        [-0.3814,  0.3274, -0.1179,  0.1605]], requires_grad=True)

- 请注意，PyTorch 中的 linear 层也是用小的随机权重初始化的；为了直接将其与上面的 `Embedding` 层进行比较，我们必须使用相同的小随机权重，这就是为什么我们在这里重新分配它们：

In [10]:
linear.weight = torch.nn.Parameter(embedding.weight.T)

- 现在我们可以对输入的 one-hot 编码表示使用 linear 层：

In [11]:
linear(onehot.float())

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)

正如我们所见，这与我们使用 embedding 层时得到的结果完全相同：

In [12]:
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- 对于第一个训练样本的 token ID，底层发生的是以下计算：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/4.png" width="450px">

**图 2.4**：Linear 层的工作原理是将 one-hot 向量与权重矩阵进行矩阵乘法。由于 one-hot 向量中只有一个元素为 1，其余全为 0，因此乘法结果实际上就是权重矩阵中对应行的值。

```
            [0, 0, 1, 0]              [第0行] [a0, a1, a2, a3]
One-hot  ×                            [第1行] [b0, b1, b2, b3]
  (ID 2)                              [第2行] [c0, c1, c2, c3] = 结果
                                      [第3行] [d0, d1, d2, d3]
```

| 方式 | 操作 | 结果 |
|------|------|------|
| Embedding 查表 | 直接取第2行 | [c0, c1, c2, c3] |
| Linear 矩阵乘法 | one-hot × 权重矩阵 | [c0, c1, c2, c3] |

> **核心结论**：对 one-hot 向量做矩阵乘法等价于直接查表，但矩阵乘法涉及大量与零的乘法，存在计算浪费。

- 对于第二个训练样本的 token ID：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/5.png" width="450px">

**图 2.5**：同样的原理应用于另一个 token ID 的 one-hot 向量与权重矩阵的乘法。

```
            [0, 0, 0, 1]              [第0行] [a0, a1, a2, a3]
One-hot  ×                            [第1行] [b0, b1, b2, b3]
  (ID 3)                              [第2行] [c0, c1, c2, c3]
                                      [第3行] [d0, d1, d2, d3] = 结果
```

> **核心结论**：每个 one-hot 向量中的 1 的位置决定了从权重矩阵中"选出"哪一行。

- 由于每个 one-hot 编码行中除了一个索引外其余全为 0（这是设计如此），这种矩阵乘法本质上等同于查找 one-hot 元素
- 在 one-hot 编码上使用矩阵乘法等价于 embedding 层的查找操作，但如果我们使用大型 embedding 矩阵，这种方法可能会效率低下，因为存在大量与零的浪费性乘法